In [ ]:
import os
# import warnings

import dask_geopandas as dg
import geopandas as gpd
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# from tqdm.notebook import tqdm
from pandarallel import pandarallel
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import linemerge, split, substring

In [ ]:
def perpendicular_segments(line: LineString, length=10):
    coords = list(line.coords)
    start, next_pt = np.array(coords[0]), np.array(coords[1])
    prev_pt, end = np.array(coords[-2]), np.array(coords[-1])

    # Tangent vectors
    v_start = next_pt - start
    v_end = end - prev_pt

    # Normalize
    v_start = v_start / np.linalg.norm(v_start)
    v_end = v_end / np.linalg.norm(v_end)

    # Rotate 90° CCW: (x, y) -> (-y, x)
    perp_start = np.array([-v_start[1], v_start[0]]) * length
    perp_end = np.array([-v_end[1], v_end[0]]) * length

    # Create perpendicular line segments
    seg_start = LineString([tuple(start - perp_start), tuple(start + perp_start)])
    seg_end = LineString([tuple(end - perp_end), tuple(end + perp_end)])

    return seg_start, seg_end

In [ ]:
def make_mask(df):
    # Get current vars
    nhd_id = df['NHDPlusID']
    width = df['WidthM']
    huc2 = df['huc2']
    huc4 = df['huc4']
    point = df['geometry']

    print(nhd_id)
    
    # Construct filepath
    huc2 = f'HUC2_{huc2}'
    huc4 = f'NHDPLUS_H_{huc4}_HU4_GDB'

    # Set data filepath
    data_path = '/nas/cee-ice/data/NHDPlus_HR/'
    file_path = os.path.join(data_path, huc2, huc4, huc4 + '.gdb')

    # Read in NHD flowlines
    basin = gpd.read_file(filename=file_path, layer='NHDFlowline',
                          engine='pyogrio')
    basin = basin.to_crs(epsg=3857) # set CRS to Pseudo-Mercator
    
    # Read in VAA for LevelPathI
    vaa = gpd.read_file(filename=file_path, layer='NHDPlusFlowlineVAA',
                    columns=['NHDPlusID', 'LevelPathI'], 
                    engine='pyogrio')
    basin = basin.merge(vaa, on='NHDPlusID')
    # level_path = basin[basin['NHDPlusID'] == nhd_id]['LevelPathI'].iloc[0]
    subset = basin[basin['NHDPlusID'] == nhd_id]['LevelPathI']
    if not subset.empty:
        level_path = subset.iloc[0]
    else:
        return None, None

    # Extract flowline
    flowline = basin[basin['NHDPlusID'] == nhd_id]
    line = flowline['geometry'].iloc[0] # DO I NEED THIS TO BE SEPARATE
    area_id = basin[basin['NHDPlusID'] == nhd_id]['WBArea_Permanent_Identifier'].iloc[0]
    
    # Find the upstream and downstream flowlines 
    touch_lines = basin[basin.geometry.intersects(line)]
    del basin
    # Filter for only those that are on the mainstem
    touch_lines = touch_lines[touch_lines['LevelPathI'] == level_path]
    
    # Merge the MultiLineStrings and descard Z coordinate
    touch_lines['temp'] = touch_lines['geometry'].apply(linemerge)
    touch_lines = touch_lines.drop(columns='geometry')
    touch_lines = touch_lines.rename(columns = {'temp': 'geometry'}).set_geometry('geometry')
    touch_lines["geometry"] = touch_lines["geometry"].apply(
        lambda geom: LineString([(x, y) for x, y, *_ in geom.coords])
    )

    # Extract geometry and merge
    touch_lines = MultiLineString(touch_lines['geometry'].to_list())
    line = linemerge(touch_lines)
    proj_dist = line.project(point)

    if not area_id:
        # Buffer flowline to get river surface polygon
        poly = line.buffer(distance=width/2, cap_style='flat')
        new_width = width
    
    else:
        area = gpd.read_file(filename=file_path, layer='NHDArea',
                      engine='pyogrio')
        area = area.to_crs(epsg=3857)
        # Extract the correct area
        temp = area[area['Permanent_Identifier'] == area_id]['geometry']
        if len(temp) == 0:
            poly = line.buffer(distance=width/2, cap_style='flat')
            new_width = width
        else: 
            poly = temp.iloc[0]
        
            # Get a more accurate width from the NHDArea
            start_dist = proj_dist - (2 * width)
            test_segment = substring(geom=line, start_dist=start_dist, end_dist=proj_dist)
            _, center = perpendicular_segments(test_segment, length=width*10)
        
            # Clip center perpendicular to area
            clipped_line = center.intersection(poly)
            new_width = clipped_line.length
    
    # Find distances
    start_dist = proj_dist - (2 * new_width)
    end_dist = proj_dist + (2 * new_width)

    # Extract segment of flowline near gauge
    segment = substring(geom=line, start_dist=start_dist, end_dist=end_dist)
    coords = list(segment.coords)
    print(coords)
    if len(coords) > 1:
        seg_start, seg_end = perpendicular_segments(segment, length=6*width)
    else:
        return None, None
    
    # Segment polygon to create mask
    temp = split(poly, seg_start)

    try:
        # Check that we segmented things in the correct direction
        if temp.geoms[1].intersects(seg_end) == True:
            mask = split(temp.geoms[1], seg_end).geoms[0]
        
        else:
            mask = split(temp.geoms[0], seg_end).geoms[1]

    except IndexError:
        # Fallback: maybe use the whole polygon, or return None
        mask = None

    print('Success!')
    return mask, new_width

In [ ]:
def get_nhd_files(filedir, huc2s):
    files = []
    huc4s = []
    for huc2 in huc2s:
        filepath = f'{filedir}{huc2}'
        for file in os.listdir(filepath):
            if not file.startswith('.'):
                files.append(os.path.join(filepath, file))
                huc4s.append(file[-28:-24])
    return files, huc4s

### First-time

In [ ]:
huc2s = ['01', '02', '03', '04', '05', '06', '07', '08', '09',
         '10', '11', '12', '13', '14', '15', '16', '17', '18']

In [ ]:
nhd_gauges = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations.parquet')
nhd_gauges = nhd_gauges.to_crs(epsg=3857)

In [ ]:
# Get IDs of sites that have discharge data
data_dir = '/nas/cee-water/cjgleason/fiona/USGS_gauges/sites/'

def find_sites_with_data(data_dir):
    site_ids = []
    for file in os.listdir(data_dir):
        site_id = file[:-8]
        site_ids.append(site_id)
    return site_ids
    
site_ids = find_sites_with_data(data_dir)

In [ ]:
nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped/HUC2_'

# Get list of NHDfiles
nhd_files, huc4s = get_nhd_files(nhd_dir, huc2s)
print(len(nhd_files))

In [ ]:
# Read in all files
# nhd = dg.read_parquet(nhd_files,
#                       columns=['NHDPlusID', 'GNIS_Name', 'LengthKM',
#                                'WidthM', 'Bin', 'StreamOrde',
#                                'Slope', 'Sinuosity', 'GageID', 'geometry']
#                      )

gdf_list = []
for i in range(len(nhd_files)):
    nhd_file = nhd_files[i]
    huc4 = huc4s[i]
    
    gdf = gpd.read_parquet(nhd_file,
                           columns=['NHDPlusID', 'GNIS_Name', 'LengthKM',
                                    'WidthM', 'Bin', 'StreamOrde',
                                    'Slope', 'Sinuosity', 'GageID', 'geometry'])
    gdf['huc4'] = huc4
    gdf['huc2'] = huc4[0:2]
    gdf_list.append(gdf)

nhd = pd.concat(gdf_list, ignore_index=True)

In [ ]:
nhd = nhd[~nhd['GageID'].isna()]
nhd = nhd.drop_duplicates(subset='NHDPlusID')

# nhd = nhd.compute()

# Subset to only sites that purport to have active stage data
nhd = nhd[nhd['GageID'].isin(ids)]

# Rename geometry for merge
nhd = nhd.rename_geometry('reach')

temp = pd.merge(left=nhd_gauges, right=nhd,#[['NHDPlusID', 'reach']],
                left_on='site_id', right_on='GageID')

def dist_to_gauge(df):
    gauge = df['geometry']
    reach = df['reach']

    return reach.distance(gauge)

temp['dist'] = temp.apply(dist_to_gauge, axis=1)
loc_off = temp[temp['dist'] > 350]['site_id'].to_list()
print(len(loc_off))

# Keep only gauges that are within 350m of a reach and have active NWIS data after download
temp = temp[~temp['site_id'].isin(loc_off)]
temp = temp[temp['site_id'].isin(site_ids)]
temp = temp.drop(columns=['GageID', 'reach'])

# Write out gauges with all metadata, no reaches > 350m from gauge, and with data from NWIS
save_dir = '/nas/cee-water/cjgleason/fiona/USGS_gauges/'
temp.to_parquet(os.path.join(save_dir, 'usgs_stations_with_stage_and_nhd_metadata.parquet'))
temp.to_file(os.path.join(save_dir, 'usgs_stations_with_stage_and_nhd_metadata.gpkg'))

for_masks = temp[['site_id', 'NHDPlusID', 'WidthM',
                  'huc2', 'huc4', 'geometry']]

### Thereafter

In [ ]:
for_masks = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata.parquet',
                             columns=['site_id', 'NHDPlusID', 'WidthM', 'huc2', 'huc4', 'geometry'])

### Make masks

In [ ]:
prior = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges.parquet')
already_masks = prior['NHDPlusID'].to_list()

for_masks = for_masks[~for_masks['NHDPlusID'].isin(already_masks)]

In [ ]:
cpus = int(os.environ.get('SLURM_CPUS_PER_TASK'))
cpus_per_task = cpus if cpus < 65 else 1
pandarallel.initialize(nb_workers=cpus_per_task)

In [ ]:
# for_masks['temp'] = for_masks.apply(make_mask, axis=1)
for_masks['temp'] = for_masks.parallel_apply(
    user_defined_function=make_mask, axis=1
)

for_masks[['reach_masks', 'new_width']] = pd.DataFrame(for_masks['temp'].tolist(),
                                                      index=for_masks.index)

for_masks = for_masks.rename(columns={'gauge_mask': 'reach_mask'}).set_geometry('reach_mask').set_crs('EPSG:3857')

for_masks.to_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_second_batch.parquet')
# for_masks = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_second_batch.parquet')

combined = pd.concat([prior, for_masks])
combined.to_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_combined.parquet')
# combined = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_combined.parquet')

In [ ]:
for i in range(len(for_masks)):
    temp = for_masks.iloc[[i]]
    temp.plot()
    plt.title(temp['site_id'].iloc[0])
    
# for_masks#[~for_masks['gauge_mask'].isna()]

In [ ]:
# for_masks['temp'] = for_masks.apply(make_mask, axis=1)
to_mask['temp'] = to_mask.parallel_apply(
    user_defined_function=make_mask, axis=1
)
to_mask[['gauge_mask', 'new_width']] = pd.DataFrame(to_mask['temp'].tolist(),
                                                      index=to_mask.index)

to_mask = to_mask.set_geometry('gauge_mask').set_crs(epsg=3857)
to_mask = to_mask.drop(columns=['temp', 'geometry'])

to_mask.to_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/gauge_reach_masks_third_batch.parquet')

In [ ]:
masks0 = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/gauge_reach_masks.parquet')
masks1 = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/gauge_reach_masks_second_batch.parquet')
masks2 = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/gauge_reach_masks_third_batch.parquet')

In [ ]:
masks0_temp = pd.merge(masks1, temp, how='inner', on='NHDPlusID')
combo = pd.concat([masks0, masks1, masks2])
mystery_site_ids = list(set(combo['site_id']) - set(for_masks['site_id']))
combo = combo[~combo['site_id'].isin(mystery_site_ids)]
combo = combo.rename(columns={'gauge_mask': 'reach_mask'}).set_geometry('reach_mask')
combo.to_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges.parquet')